In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()


Instructions for updating:
non-resource variables are not supported in the long term


선형회귀는 알려진 데이터를 사용해서 알 수 없는 데이터를 예측하는 데이터 분석 기법이다.  
알 수 없는 변수 또는 종속 변수와 알려진 변수 또는 독립 변수를 선형 방정식으로 수학적으로 모델링한다.

학습 데이터를 만든다.

In [2]:
xData = [1, 2, 3, 4, 5, 6, 7, 8] # 근무 시간, 데이터
yData = [25000, 55000, 75000, 110000, 128000, 155000, 180000, 232500] # 근무 시간에 따른 매출 금액, 실제값, 관측값, 답, 레이블

$y = ax + b$를 만족시키는 $a$(기울기, 가중치)와 $b$(y절편, 바이어스)를 정해야 한다.  
과적합을 방지하기 위해서 난수를 발생시켜 가중치와 바이어스를 정한다.  
과적합이란 학습한 데이터에서는 높은 정확도를 보이지만 학습한 데이터 이외의 데이터에서는 낮은 정확도를 보이는 문제점을 말한다.

random_uniform() 메소드로 최소값과 최대값 사이에서 지정한 개수의 난수를 발생시킨다.  
random_uniform([난수의 개수], 최소값, 최대값)

In [3]:
a = tf.Variable(tf.random_uniform([1], -100, 100)) # 가중치, -100 ~ 100 사이의 난수
b = tf.Variable(tf.random_uniform([1], -100, 100)) # 바이어스, -100 ~ 100 사이의 난수
print('a = {}, b = {}'.format(a, b))

a = <tf.Variable 'Variable:0' shape=(1,) dtype=float32_ref>, b = <tf.Variable 'Variable_1:0' shape=(1,) dtype=float32_ref>


tensorflow 변수를 초기화하고 난수로 발생시킨 가중치와 바이어스를 확인한다.

In [4]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())
print('a = {}, b = {}'.format(sess.run(a), sess.run(b)))

a = [-78.19641], b = [-52.600815]


근무 시간과 근무 시간에 따른 매출 금액을 기억할 placeholder를 선언한다.

In [5]:
x = tf.placeholder(dtype=tf.float32) # 근무 시간을 기억할 placeholder를 선언한다.
y = tf.placeholder(dtype=tf.float32) # 근무 시간에 따른 매출 금액(실제값)을 기억할 placeholder를 선언한다.

1차 방정식 형태의 가설($y = ax + b$)을 세운다.

In [6]:
# 1차 방정식 형태의 가설, Y => 예측값
Y = a * x + b

오차(손실(loss), 비용(cost)) 함수를 정의한다.

In [7]:
# 평균 제곱법을 이용하는 오차함수
# 실제값(yData => y라는 placeholder)과 예측값(Y)의 편차에 대한 제곱의 평균을 이용한다.
# square() 메소드로 tensorflow는 제곱값을 계산한다.
# reduce_mean() 메소드로 tensorflow는 평균을 계산한다.
cost = tf.reduce_mean(tf.square(Y - y))

경사 하강법 알고리즘을 이용해서 오차 함수의 결과값을 가장 작게만드는 방향으로 학습하도록 정의한다.

In [8]:
# 경사 하강법 알고리즘의 학습율(learning rate)을 설정한다.
learning_rate = tf.Variable(0.001)
# train.GradientDescentOptimizer() 클래스의 인수로 학습율을 넘겨서 경사 하강법 알고리즘을 이용해 오차를 계산한다.
# minimize() 메소드의 인수로 오차 함수를 넘겨서 경사 하강법에 의해 계산되는 오차의 최소값을 찾는다.
train = tf.train.GradientDescentOptimizer(learning_rate).minimize(cost)

학습시킨다.

In [21]:
# 세션을 만들고 변수를 초기화시킨다.
sess = tf.Session()
sess.run(tf.global_variables_initializer())

for i in range(20001):
    # 오차 함수의 결과를 가장 작게 만드는 학습을 할 수 있도록 placeholder에 데이터를 넣어주고 학습시킨다.
    sess.run(train, feed_dict={x: xData, y:yData})
    
    # 일정 epoch(작업 단위, 학습을 1번 실행하는)마다 중간 결과를 출력한다.
    if i % 2000 == 0:
        print(
            'epoch: {:5d}, cost: {:14.2f}, a: {:9.2f}, b: {:9.2f}'.format(
                i, sess.run(cost, feed_dict={x: xData, y:yData}), sess.run(a)[0], sess.run(b)[0]
            )
        )
# ===== 학습끝

print('최적의 가중치: {:9.2f}, 최적의 바이어스: {:9.2f}'.format(sess.run(a)[0], sess.run(b)[0]))
print('9시간 근무했을 때 예상 매출 금액: {:,.0f}원'.format(sess.run(a)[0] * 9 + sess.run(b)[0]))

epoch:     0, cost: 16705624064.00, a:   1337.38, b:    151.21
epoch:  2000, cost:    60423616.00, a:  27034.63, b:   -730.39
epoch:  4000, cost:    57354040.00, a:  27457.58, b:  -3108.33
epoch:  6000, cost:    56732548.00, a:  27647.87, b:  -4178.24
epoch:  8000, cost:    56606736.00, a:  27733.50, b:  -4659.64
epoch: 10000, cost:    56581268.00, a:  27772.03, b:  -4876.26
epoch: 12000, cost:    56576108.00, a:  27789.36, b:  -4973.72
epoch: 14000, cost:    56575044.00, a:  27797.17, b:  -5017.63
epoch: 16000, cost:    56574812.00, a:  27800.70, b:  -5037.41
epoch: 18000, cost:    56574796.00, a:  27802.26, b:  -5046.24
epoch: 20000, cost:    56574776.00, a:  27802.95, b:  -5050.14
최적의 가중치:  27802.95, 최적의 바이어스:  -5050.14
9시간 근무했을 때 예상 매출 금액: 245,176원


In [19]:
for i in range(9, 25):
    # print('{:2d}시간 근무했을 때 예상 매출 금액: {:,.0f}원'.format(i, sess.run(a)[0] * i + sess.run(b)[0]))
    print('{:2d}시간 근무했을 때 예상 매출 금액: {:,.0f}원'.format(i, sess.run(Y, feed_dict={x: [i]})[0]))

 9시간 근무했을 때 예상 매출 금액: 245,176원
10시간 근무했을 때 예상 매출 금액: 272,979원
11시간 근무했을 때 예상 매출 금액: 300,782원
12시간 근무했을 때 예상 매출 금액: 328,585원
13시간 근무했을 때 예상 매출 금액: 356,388원
14시간 근무했을 때 예상 매출 금액: 384,191원
15시간 근무했을 때 예상 매출 금액: 411,994원
16시간 근무했을 때 예상 매출 금액: 439,797원
17시간 근무했을 때 예상 매출 금액: 467,600원
18시간 근무했을 때 예상 매출 금액: 495,403원
19시간 근무했을 때 예상 매출 금액: 523,206원
20시간 근무했을 때 예상 매출 금액: 551,009원
21시간 근무했을 때 예상 매출 금액: 578,812원
22시간 근무했을 때 예상 매출 금액: 606,615원
23시간 근무했을 때 예상 매출 금액: 634,418원
24시간 근무했을 때 예상 매출 금액: 662,221원
